In [2]:
from pydantic import BaseModel, Field
from typing import List, Optional, Literal, Dict

# --- Section Models ---

class EducationItem(BaseModel):
    institution: str = Field(..., description="Name of the university or school.")
    degree: str = Field(..., description="Degree obtained (e.g., 'B.Tech', 'Master of Science').")
    field_of_study: str = Field(default="", description="Major or specialization.")
    location: str = Field(default="", description="City, Country.")
    start_date: str = Field(..., description="Start date (e.g., '2019').")
    end_date: str = Field(..., description="End date or 'Present'.")
    grade: Optional[str] = Field(None, description="GPA, CGPA, or Percentage if listed.")

class WorkExperienceItem(BaseModel):
    role: str = Field(..., description="Job title.")
    company: str = Field(..., description="Name of the company.")
    location: str = Field(default="Remote", description="City, Country or 'Remote'.")
    start_date: str = Field(..., description="Start date.")
    end_date: str = Field(..., description="End date or 'Present'.")
    description_bullets: List[str] = Field(..., description="Achievements and responsibilities.")
    tech_stack: List[str] = Field(default=[], description="Tools/Languages used in this specific role.")

class ProjectItem(BaseModel):
    name: str = Field(..., description="Project name.")
    description: str = Field(..., description="Brief summary.")
    tech_stack: List[str] = Field(..., description="Technologies used.")
    url: Optional[str] = Field(None, description="Link to code or demo.")
    bullets: List[str] = Field(default=[], description="Key outcomes or features.")

class CertificationItem(BaseModel):
    name: str = Field(..., description="Name of the certification (e.g., 'AWS Certified Solutions Architect').")
    issuer: str = Field(default="", description="Organization (e.g., 'Amazon', 'Google').")
    date: Optional[str] = Field(None, description="Date obtained or expiry.")

class VolunteerItem(BaseModel):
    role: str = Field(..., description="Role title (e.g., 'Mentor', 'Volunteer').")
    organization: str = Field(..., description="Organization name.")
    description: Optional[str] = Field(None, description="Brief description of impact.")

class AwardItem(BaseModel):
    title: str = Field(..., description="Name of the award or honor.")
    issuer: str = Field(default="", description="Who gave the award.")
    date: Optional[str] = Field(None, description="Date received.")

# --- Main Resume Extraction Model ---

class ResumeMetaData(BaseModel):
    # 1. Contact Information
    full_name: str = Field(..., description="Candidate's full name.")
    email: str = Field(..., description="Email address.")
    phone: str = Field(default="", description="Phone number.")
    location: str = Field(default="", description="City, State/Country.")
    linkedin_url: Optional[str] = Field(None, description="LinkedIn profile URL.")
    github_url: Optional[str] = Field(None, description="GitHub or Portfolio URL.")

    # 2. Summary/Objective
    summary: str = Field(default="", description="Professional summary or objective statement.")

    # 3. Work Experience
    work_experience: List[WorkExperienceItem] = Field(default=[], description="Professional history.")

    # 4. Technical Skills
    skills: Dict[str, List[str]] = Field(
        ..., 
        description="Skills categorized by type (e.g., {'Languages': ['Python'], 'Cloud': ['AWS']} )."
    )

    # 5. Education
    education: List[EducationItem] = Field(..., description="Educational background.")

    # --- Optional/Enhancing Sections ---
    
    # 6. Projects
    projects: List[ProjectItem] = Field(default=[], description="Personal or academic projects.")

    # 7. Certifications
    certifications: List[CertificationItem] = Field(default=[], description="Professional certifications.")

    # 8. Awards & Honors
    awards: List[AwardItem] = Field(default=[], description="Awards, hackathon wins, or honors.")

    # 9. Volunteer Experience
    volunteer_experience: List[VolunteerItem] = Field(default=[], description="Volunteering and community service.")

    # 10. Interests/Hobbies
    interests: List[str] = Field(default=[], description="List of personal interests or hobbies.")

    # --- Meta-Analysis (Computed Fields for Agents) ---
    total_years_experience: float = Field(..., description="Total years of professional experience.")


class JobMetadata(BaseModel):
    # --- Core Job Data ---
    job_title: str = Field(..., description="Official name of the position (e.g., 'Senior Backend Engineer').")
    company_name: str = Field(..., description="Name of the hiring entity.")
    location: str = Field(..., description="City e.g. Delhi, State e.g. Bangalore, or 'Remote'.")
    employment_type: Literal["Full-time", "Part-time", "Contract", "Internship", "Freelance", "Unknown"] = Field(..., description="Type of employment.")
    salary_range: str = Field(..., description="Extracted compensation (e.g., '$120k-$150k', '20 LPA', '₹20000/month'). Use 'Not Disclosed' if missing.")
    
    # --- Job Identification & Summary ---
    department: str = Field(default="Unknown", description="e.g., 'Engineering', 'Sales'.")
    reporting_to: str = Field(default="Unknown", description="Manager title (e.g., 'CTO', 'Engineering Manager').")
    job_summary: str = Field(..., description="A 3-4 sentence high-level overview of why the job exists.")
    company_overview: str = Field(default="", description="Brief mission or culture description.")

    # --- Structured Requirements ---
    experience_level: Literal["Entry", "Mid", "Senior", "Lead", "Executive", "Unknown"] = Field(..., description=" inferred seniority level.")
    min_education: str = Field(default="Not Specified", description="e.g., 'Bachelor’s in CS', 'Master’s'.")
    required_skills: List[str] = Field(..., description="List of MANDATORY technical or soft skills.")
    preferred_skills: List[str] = Field(default=[], description="Nice-to-have skills that make a candidate stand out.")
    
    # --- Duties & Conditions ---
    duties_responsibilities: List[str] = Field(..., description="List of essential functions or tasks.")
    work_mode: Literal["Remote", "On-Site", "Hybrid"] = Field(..., description="Working environment.")
    benefits: List[str] = Field(default=[], description="Perks like 'Health Insurance', 'Stock Options', 'Free Food'.")

class MetaDataExtraction(BaseModel):
    job_description: Optional[JobMetadata] = Field(description="Extract Job Metadata")
    resume: Optional[ResumeMetaData] = Field(description="Extract Resume Metadata")


In [6]:
import os 
os.getcwd()
os.chdir("d:\\GyanPrakashKuswaha\\GenAI\\InternOps")

In [16]:
from langchain_google_genai import ChatGoogleGenerativeAI
from dotenv import load_dotenv

load_dotenv()

True

In [17]:
llm = ChatGoogleGenerativeAI(model="gemini-2.5-flash")

In [19]:
jd = """
Job Description
Job Title: Data Science & AI Intern  

Company: GEODISHA 

Location: Hyderabad - Onsite 

Duration: 1- 4 Months 

About GEODISHA 

At GEODISHA, we are at the forefront of Data Analytics and AI, leveraging data to solve  complex problems and drive innovation. Our team is a dedicated group of researchers, engineers,  and strategists who believe in the power of technology. We are passionately committed to  developing cutting-edge AI that is not only powerful but also ethical, transparent, and fair. We're  looking for the next generation of innovators to join us. 

The Opportunity: This Isn't Your Typical Internship 

We are seeking truly exceptional interns to join our core Data & AI team. This is a unique  opportunity to move beyond theory and apply your skills to high-impact, real-world challenges  across the full data lifecycle. 

You won't be on the sidelines. You'll be paired with a senior mentor and embedded directly into  projects at the intersection of data engineering, data analytics, artificial intelligence, and human  behavior. We are looking for a candidate who is not just an outstanding programmer, but a  critical thinker who understands that great AI starts with great data, is passionate about the  "why" behind the data, and sees the critical importance of building Responsible AI. 

What You’ll Do (Key Responsibilities): 

∉ Assist in developing AI-driven recommendation engines and personalization workflows. ∉ Analyze structured and unstructured datasets to derive insights and patterns. ∉ Work with senior team members on customer segmentation, predictive modeling, and LLM integrated analytics tools. 

∉ Learn and support the development of modular data pipelines and models using industry  tools. 

∉ Use Python, SQL, scikit-learn, and other libraries for experimentation and development. ∉ Present findings through visualization tools such as seaborn, matplotlib, or BI tools like  Power BI. 

∉ Document code, experiments, and insights for further development. 

Who You Are (Our Ideal Candidate):

We are looking for a rising star who is driven, curious, and eager to make a tangible impact. 

Core Requirements: 

∉ Currently pursuing or recently completed a degree in Data Science, Computer Science,  Statistics, Mathematics, or related field. 

∉ Good understanding of Python and SQL (R is a plus). 

∉ Familiarity with basic ML algorithms: classification, regression, clustering. ∉ Exposure to libraries such as pandas, NumPy, scikit-learn, etc. 

∉ Interest in GenAI/NLP concepts like transformers or LLMs is a bonus. ∉ Strong analytical mindset, curiosity, and willingness to learn. 

Passion & Interest (What Sets You Apart): 

∉ A genuine and demonstrable passion for the entire Data Analytics space, from robust  engineering to insightful analysis. 

∉ Academic or project-based exposure to Behavioral Analytics or computational social  science. 

∉ A strong, well-articulated interest in the field of Responsible AI, ethics, and algorithmic  fairness 

Bonus Points: 

∉ Exposure to tools like TensorFlow, Hugging Face, or BI dashboards. 

∉ Basic understanding of cloud platforms (AWS/GCP/Azure), Git, or APIs. ∉ Academic projects, personal experiments, or GitHub repositories demonstrating interest in  AI/ML. 

What You’ll Gain: 

● Hands-on experience in a startup environment working on cutting-edge AdTech & MarTech  products. 

● Mentorship from senior Data Science and AI professionals. 

● A chance to convert the internship into a full-time position based on performance.

Type of Opportunity: Internship 

Job Title: Data Science & AI Intern


"""

resume = """
Gyan Prakash Kushwaha
/githubGitHub| /linkedinLinkedIn| /gl⌢beKaggle| /envel⌢pegyanprakash.sde| ♂¶obile+91 9575765381
SUMMARY
AI Engineering Undergraduate (IIT Madras) with strong DSA fundamentals (200+ LeetCodeproblems). Experi-
enced in buildingagentic RAG workflowsand end-to-end ML systems usingLangGraph, FastAPI, and Vector
DBs. Proficient in reducing inference latency and deploying scalable AI solutions. Seeking an SDE/AI internship to
leverage skills in Generative AI and backend optimization.
EDUCATION
Indian Institute of Technology (IIT), MadrasChennai, Tamil Nadu
BS in Data Science and Applications; CGPA: 8.18/10 2023 – 2027
SKILLS
Programming Languages Python, JavaScript (ES6+), Java, SQL
Backend Frameworks Flask, FastAPI, Node.js
Frontend Technologies React, Vue.js, Bootstrap, Tailwind CSS
Databases & Storage Systems PostgreSQL, SQLite, DuckDB, Redis
Vector Databases & Search FAISS, ChromaDB
Machine Learning & NLP NumPy, Pandas, Scikit-learn, PyTorch, TensorFlow, NLTK
Data Visualization Power BI, Matplotlib, Seaborn, Plotly
Generative AI & LLM Frameworks LangChain, LangGraph, LangSmith, AGNO
DevOps & Systems Linux, Git, GitHub, Celery
Cloud Platforms AWS (EC2, S3, Lambda), GCP
PROJECTS
YouTube RAG Chatbot|LangGraph, FastAPI, FAISS, AsyncSQLite, Vue.js, Gemini (Flash-Lite) Repo Link
– ReducedTime-to-First-Token (TTFT) by 90%(from 5s to<500ms) by engineering a real-time async streaming
pipeline withFastAPIandLangGraph, significantly improving user perceived responsiveness.
– Architected astateful RAG workflowusingLangGraphandAsyncSQLiteto orchestrate persistent, multi-turn
conversations, enabling context coherence across1-hour+ video transcripts.
– Increased RAG response accuracy by implementingMaximal Marginal Relevance (MMR)search, filtering
90% of redundant contextfrom long-form audio transcripts to minimizeLLM hallucinations.
CropGuardian-AI|LangGraph, FastAPI, Pydantic, Gemini 1.5 Pro, OpenMeteo Repo Link
– Architected amulti-agent workflowusingLangGraphto orchestrate3 specialized agents(Vision, Weather,
Agronomy), automating the complex reasoning chain from image analysis to actionable advice with<3s latency.
– Implemented strictStructured Outputenforcement usingPydanticandGemini 1.5 Pro, achieving100%
schema compliancefor JSON responses and eliminating parsing errors in the frontend application.
– Designed acontext-aware reasoning enginethat grounds visual diagnosis in real-time environmental data
(OpenMeteo API), generating location-specific farming action plans based on7-day weather forecasts.
Customer Churn Predictor|Python, Scikit-learn, XGBoost, MLflow, Streamlit Repo Link
– Developed a modular machine learning pipeline to process100,000 customer records, benchmarking perfor-
mance across4 ensemble algorithms(XGBoost, Random Forest, AdaBoost, Gradient Boosting) to identify op-
timal churn predictors.
– Engineered a robust training workflow integrated withMLflowto systematically track model metrics, parameters,
and version history, ensuringreproducibilityacross experimentation cycles.
– Deployed the best-performing model as an interactive web application usingStreamlit, enabling non-technical
stakeholders to input customer demographics and generatereal-time churn risk assessments.
Movie Recommender System|Python, Scikit-Learn, Streamlit, Pandas Repo Link
–Developed a Content-Based Movie Recommender SystemusingPythonandStreamlit, processing a dataset
of4,800+ moviesto generate personalized top-10 viewing suggestions.
–Engineered a feature extraction pipelinewithPandasandScikit-Learn, transforming unstructured metadata
(genres, cast, crew) into a5,000-featureBag-of-Words model to calculateCosine Similarityscores.
–Deployed an interactive web applicationintegrating theTMDB APIto fetch real-time posters, utilizingPickle
serialization to optimize data loading and deliver recommendations efficiently.
ACHIEVEMENTS
•Kaggle Expert: Top 4% globally (Rank 341), 1 Silver & 9 Bronze medals; datasets with 22K+ views and 5.6K+
downloads
•LeetCode: Solved 219+ DSA problems (100+ Medium)
•HackerRank: 5⋆Gold Badge (SQL)

"""

In [22]:
from app.prompts import ExtractionPrompt

extractor_agent = llm.with_structured_output(MetaDataExtraction)

# Ensure JobMetaDataExtractionPrompt is defined in your prompts.py, 
# otherwise use a simple template here.
agent_prompt = ExtractionPrompt.EXTRACTION_PROMPT.format(
    job_description = jd,
    resume_text = resume
)

response = extractor_agent.invoke(agent_prompt)
agent_prompt

'System: \n        ** Task-1 **\n         You are an expert recruitment data analyst. Extract job details from the provided text into the structured schema.\n\n        Follow these extraction rules:\n        1. **Inference:** If fields like \'experience_level\' or \'work_mode\' are not explicitly labeled, infer them from context (e.g., if the text says "5+ years experience", mark experience_level as "Senior").\n        2. **Missing Data:** If a mandatory field is strictly missing and cannot be inferred, use "Unknown" or "Not Disclosed" as applicable.\n        3. **Skills:** Extract \'required_skills\' as distinct, atomic strings (e.g., split "Python/Django" into "Python", "Django").\n        4. **Summary:** For \'job_summary\', synthesize a concise 3-4 sentence overview from the description; do not just copy the first paragraph.\n        \n        ** task-2 **\n        You are an expert Resume Parser. Your job is to extract candidate details into a precise, structured JSON format.\n\n 

In [23]:
response

MetaDataExtraction(job_description=JobMetadata(job_title='Data Science & AI Intern', company_name='GEODISHA', location='Hyderabad', employment_type='Internship', salary_range='Not Disclosed', department='Data & AI', reporting_to='Senior Mentor', job_summary='GEODISHA is seeking exceptional Data Science & AI Interns to join its core Data & AI team. This internship offers a unique opportunity to apply skills to high-impact, real-world challenges across the full data lifecycle. Interns will work on projects at the intersection of data engineering, data analytics, artificial intelligence, and human behavior, focusing on responsible AI development. The role involves developing AI-driven solutions, analyzing datasets, and supporting the development of data pipelines and models.', company_overview='At GEODISHA, we are at the forefront of Data Analytics and AI, leveraging data to solve complex problems and drive innovation. Our team is a dedicated group of researchers, engineers, and strategis

In [3]:
res = MetaDataExtraction(job_description=JobMetadata(job_title='Data Science & AI Intern', company_name='GEODISHA', location='Hyderabad', employment_type='Internship', salary_range='Not Disclosed', department='Data & AI', reporting_to='Senior Mentor', job_summary='GEODISHA is seeking exceptional Data Science & AI Interns to join its core Data & AI team. This internship offers a unique opportunity to apply skills to high-impact, real-world challenges across the full data lifecycle. Interns will work on projects at the intersection of data engineering, data analytics, artificial intelligence, and human behavior, focusing on responsible AI development. The role involves developing AI-driven solutions, analyzing datasets, and supporting the development of data pipelines and models.', company_overview='At GEODISHA, we are at the forefront of Data Analytics and AI, leveraging data to solve complex problems and drive innovation. Our team is a dedicated group of researchers, engineers, and strategists who believe in the power of technology. We are passionately committed to developing cutting-edge AI that is not only powerful but also ethical, transparent, and fair.', experience_level='Entry', min_education='Currently pursuing or recently completed a degree in Data Science, Computer Science, Statistics, Mathematics, or related field.', required_skills=['Python', 'SQL', 'Machine Learning algorithms', 'Classification', 'Regression', 'Clustering', 'Pandas', 'NumPy', 'Scikit-learn', 'Analytical mindset', 'Curiosity', 'Willingness to learn'], preferred_skills=['R', 'Generative AI', 'NLP', 'Transformers', 'LLMs', 'Behavioral Analytics', 'Computational Social Science', 'Responsible AI', 'Ethics', 'Algorithmic Fairness', 'TensorFlow', 'Hugging Face', 'BI Dashboards', 'AWS', 'GCP', 'Azure', 'Git', 'APIs'], duties_responsibilities=['Assist in developing AI-driven recommendation engines and personalization workflows.', 'Analyze structured and unstructured datasets to derive insights and patterns.', 'Work with senior team members on customer segmentation, predictive modeling, and LLM integrated analytics tools.', 'Learn and support the development of modular data pipelines and models using industry tools.', 'Use Python, SQL, scikit-learn, and other libraries for experimentation and development.', 'Present findings through visualization tools such as seaborn, matplotlib, or BI tools like Power BI.', 'Document code, experiments, and insights for further development.'], work_mode='On-Site', benefits=['Hands-on experience in a startup environment', 'Mentorship from senior Data Science and AI professionals', 'Chance to convert the internship into a full-time position based on performance']), resume=ResumeMetaData(full_name='Gyan Prakash Kushwaha', email='gyanprakash.sde', phone='+91 9575765381', location='Chennai, Tamil Nadu', linkedin_url=None, github_url=None, summary='AI Engineering Undergraduate (IIT Madras) with strong DSA fundamentals (200+ LeetCode problems). Experienced in building agentic RAG workflows and end-to-end ML systems using LangGraph, FastAPI, and Vector DBs. Proficient in reducing inference latency and deploying scalable AI solutions. Seeking an SDE/AI internship to leverage skills in Generative AI and backend optimization.', work_experience=[], skills={'Programming Languages': ['Python', 'JavaScript (ES6+)', 'Java', 'SQL'], 'Backend Frameworks': ['Flask', 'FastAPI', 'Node.js'], 'Frontend Technologies': ['React', 'Vue.js', 'Bootstrap', 'Tailwind CSS'], 'Databases & Storage Systems': ['PostgreSQL', 'SQLite', 'DuckDB', 'Redis'], 'Vector Databases & Search': ['FAISS', 'ChromaDB'], 'Machine Learning & NLP': ['NumPy', 'Pandas', 'Scikit-learn', 'PyTorch', 'TensorFlow', 'NLTK'], 'Data Visualization': ['Power BI', 'Matplotlib', 'Seaborn', 'Plotly'], 'Generative AI & LLM Frameworks': ['LangChain', 'LangGraph', 'LangSmith', 'AGNO'], 'DevOps & Systems': ['Linux', 'Git', 'GitHub', 'Celery'], 'Cloud Platforms': ['AWS (EC2, S3, Lambda)', 'GCP']}, education=[EducationItem(institution='Indian Institute of Technology (IIT), Madras', degree='BS', field_of_study='Data Science and Applications', location='Chennai, Tamil Nadu', start_date='2023', end_date='2027', grade='8.18/10 CGPA')], projects=[ProjectItem(name='YouTube RAG Chatbot', description='Engineered a real-time async streaming pipeline using FastAPI and LangGraph to reduce Time-to-First-Token (TTFT) by 90% and architected a stateful RAG workflow with AsyncSQLite for persistent, multi-turn conversations. Implemented Maximal Marginal Relevance (MMR) search to increase RAG response accuracy.', tech_stack=['LangGraph', 'FastAPI', 'FAISS', 'AsyncSQLite', 'Vue.js', 'Gemini (Flash-Lite)'], url='Repo Link', bullets=['Reduced Time-to-First-Token (TTFT) by 90% (from 5s to <500ms) by engineering a real-time async streaming pipeline with FastAPI and LangGraph, significantly improving user perceived responsiveness.', 'Architected a stateful RAG workflow using LangGraph and AsyncSQLite to orchestrate persistent, multi-turn conversations, enabling context coherence across 1-hour+ video transcripts.', 'Increased RAG response accuracy by implementing Maximal Marginal Relevance (MMR) search, filtering 90% of redundant context from long-form audio transcripts to minimize LLM hallucinations.']), ProjectItem(name='CropGuardian-AI', description='Architected a multi-agent workflow using LangGraph to orchestrate 3 specialized agents (Vision, Weather, Agronomy), automating complex reasoning from image analysis to actionable advice. Implemented strict Structured Output enforcement using Pydantic and Gemini 1.5 Pro, and designed a context-aware reasoning engine.', tech_stack=['LangGraph', 'FastAPI', 'Pydantic', 'Gemini 1.5 Pro', 'OpenMeteo'], url='Repo Link', bullets=['Architected a multi-agent workflow using LangGraph to orchestrate 3 specialized agents (Vision, Weather, Agronomy), automating the complex reasoning chain from image analysis to actionable advice with <3s latency.', 'Implemented strict Structured Output enforcement using Pydantic and Gemini 1.5 Pro, achieving 100% schema compliance for JSON responses and eliminating parsing errors in the frontend application.', 'Designed a context-aware reasoning engine that grounds visual diagnosis in real-time environmental data (OpenMeteo API), generating location-specific farming action plans based on 7-day weather forecasts.']), ProjectItem(name='Customer Churn Predictor', description='Developed a modular machine learning pipeline to process 100,000 customer records, benchmarking performance across 4 ensemble algorithms. Engineered a robust training workflow integrated with MLflow for reproducibility and deployed the best-performing model as an interactive web application using Streamlit.', tech_stack=['Python', 'Scikit-learn', 'XGBoost', 'MLflow', 'Streamlit'], url='Repo Link', bullets=['Developed a modular machine learning pipeline to process 100,000 customer records, benchmarking performance across 4 ensemble algorithms (XGBoost, Random Forest, AdaBoost, Gradient Boosting) to identify optimal churn predictors.', 'Engineered a robust training workflow integrated with MLflow to systematically track model metrics, parameters, and version history, ensuring reproducibility across experimentation cycles.', 'Deployed the best-performing model as an interactive web application using Streamlit, enabling non-technical stakeholders to input customer demographics and generate real-time churn risk assessments.']), ProjectItem(name='Movie Recommender System', description='Developed a Content-Based Movie Recommender System using Python and Streamlit, processing a dataset of 4,800+ movies. Engineered a feature extraction pipeline with Pandas and Scikit-Learn, and deployed an interactive web application integrating the TMDB API.', tech_stack=['Python', 'Scikit-Learn', 'Streamlit', 'Pandas'], url='Repo Link', bullets=['Developed a Content-Based Movie Recommender System using Python and Streamlit, processing a dataset of 4,800+ movies to generate personalized top-10 viewing suggestions.', 'Engineered a feature extraction pipeline with Pandas and Scikit-Learn, transforming unstructured metadata (genres, cast, crew) into a 5,000-feature Bag-of-Words model to calculate Cosine Similarity scores.', 'Deployed an interactive web application integrating the TMDB API to fetch real-time posters, utilizing Pickle serialization to optimize data loading and deliver recommendations efficiently.'])], certifications=[], awards=[AwardItem(title='Kaggle Expert', issuer='Kaggle', date=None), AwardItem(title='Solved 219+ DSA problems (100+ Medium)', issuer='LeetCode', date=None), AwardItem(title='5⋆Gold Badge (SQL)', issuer='HackerRank', date=None)], volunteer_experience=[], interests=[], total_years_experience=0.0))

In [7]:
from app.database import get_db_connection

In [ ]:
URI = "postgresql://postgres:password@db:5432/internops?sslmode=disable"

In [ ]:
res_meta = res["resume_metadata"]
# resume_query = """
# INSERT INTO resume_parsed_data (
#     analysis_id,
#     full_name, email, phone, location, 
#     linkedin_url, github_url, summary, total_years_experience,
#     education, work_experience, skills, projects,
#     certifications, awards, volunteer_experience, interests
# )
# VALUES (
#     %s, 
#     %s, %s, %s, %s, 
#     %s, %s, %s, %s,
#     %s, %s, %s, %s, 
#     %s, %s, %s, %s
# )
# """

# # 3. Execute with JSON serialization for nested lists/dicts
# cur.execute(resume_query, (
#     analysis_id,
#     res_meta["full_name"],
#     res_meta["email"],
#     res_meta["phone"],
#     res_meta["location"],
    
#     res_meta["linkedin_url"],
#     res_meta["github_url"],
#     res_meta["summary"],
#     res_meta["total_years_experience"],

#     # Complex nested fields must be dumped to JSON strings
#     res_meta["education"],
#     res_meta["work_experience"],
#     res_meta["skills"],
#     res_meta["projects"],
    
#     res_meta["certifications"],
#     res_meta["awards"],
#     res_meta["volunteer_experience"],
#     res_meta["interests"]
# ))


In [5]:
res.job_description

JobMetadata(job_title='Data Science & AI Intern', company_name='GEODISHA', location='Hyderabad', employment_type='Internship', salary_range='Not Disclosed', department='Data & AI', reporting_to='Senior Mentor', job_summary='GEODISHA is seeking exceptional Data Science & AI Interns to join its core Data & AI team. This internship offers a unique opportunity to apply skills to high-impact, real-world challenges across the full data lifecycle. Interns will work on projects at the intersection of data engineering, data analytics, artificial intelligence, and human behavior, focusing on responsible AI development. The role involves developing AI-driven solutions, analyzing datasets, and supporting the development of data pipelines and models.', company_overview='At GEODISHA, we are at the forefront of Data Analytics and AI, leveraging data to solve complex problems and drive innovation. Our team is a dedicated group of researchers, engineers, and strategists who believe in the power of tech

In [2]:
from pydantic import BaseModel

In [6]:
class User(BaseModel):
    email: str
    password: str
    
user = User(email= "gyan022", password = "password")
type(user.email)

str